# SOFR and Fed Funds curves, and a risk template

1. Calibrate `USD.SOFR.1D` from deposits and par swaps.
2. Register `USD.FEDFUND.1D` as SOFR plus a spread, and calibrate the spread from Fed Funds par swaps.
3. Price a small book and report its risk against a template of par swaps on both curves.

Run from the repo root with `PYTHONPATH=src`.

In [1]:
import numpy as np
import pandas as pd

from finance.dates import Date
from finance.instruments.resolution import Swap
from finance.markets.context import MarketContext
from finance.markets.curves import CurveInterpolator
from finance.pricing.calibration import CurveCalibrator, CurveDefinition, GlobalSolver, deposit_helper, swap_helper
from finance.pricing.pricers import SwapPricer
from finance.pricing.risk import RiskEngine, RiskTemplate

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

AS_OF = Date(2026, 10, 16)

## 1. The SOFR curve

Each helper carries one quote. The definition names the key being built and says the same curve
also serves `USD.STDCSA`, the discount key the swaps name. `calibrate` returns the solved curve
and a market holding it.

In [2]:
sofr_quotes = {"1M": 0.0430, "3M": 0.0425}
sofr_swaps = {"1Y": 0.0410, "2Y": 0.0385, "3Y": 0.0375, "5Y": 0.0370, "7Y": 0.0372, "10Y": 0.0378, "20Y": 0.0390, "30Y": 0.0388}

sofr_helpers = [deposit_helper(rate=r, tenor=t, as_of=AS_OF) for t, r in sofr_quotes.items()] + [
    swap_helper(rate=r, tenor=t, as_of=AS_OF) for t, r in sofr_swaps.items()
]
sofr_definition = CurveDefinition("USD.SOFR.1D", CurveInterpolator.LogLinearDF, discount_keys=("USD.STDCSA",))
sofr_result = CurveCalibrator(sofr_helpers, GlobalSolver(), sofr_definition).calibrate(MarketContext(AS_OF))

market = sofr_result.market
print(market)
print(f"converged: {sofr_result.solver_result.converged}, max |residual| = {abs(sofr_result.residuals).max():.1e}")
pd.DataFrame({
    "pillar": sofr_result.pillar_dates,
    "zero rate %": 100 * market.yield_curve("USD.SOFR.1D").zero_rate(sofr_result.pillar_dates),
    "df": market.yield_curve("USD.SOFR.1D").discount_factor(sofr_result.pillar_dates),
})

MarketContext(as_of=2026-10-16, curves=['USD.SOFR.1D', 'USD.STDCSA'], frozen)
converged: True, max |residual| = 2.3e-15


,pillar,zero rate %,df
0,2026-11-20,4.35,1.00
1,2027-01-20,4.29,0.99
2,2027-10-20,4.08,0.96
3,2028-10-20,3.83,0.93
4,2029-10-22,3.73,0.89
5,2031-10-20,3.68,0.83
6,2033-10-20,3.70,0.77
7,2036-10-20,3.77,0.69
8,2046-10-22,3.91,0.46
9,2056-10-20,3.87,0.31


## 2. The Fed Funds curve as SOFR plus a spread

There is no basis-swap helper yet, so the Fed Funds quotes are par rates of fixed-vs-Fed-Funds
swaps, discounted at SOFR like everything else. The curve is registered as the SOFR object plus a
flat spread curve on the instruments' pillars; calibrating the Fed Funds key then solves only the
spread, and the curve stays built on SOFR.

In [3]:
fedfund_swaps = {"1Y": 0.0404, "2Y": 0.0379, "3Y": 0.0369, "5Y": 0.0364, "10Y": 0.0372}
fedfund_helpers = [swap_helper(rate=r, tenor=t, as_of=AS_OF, rate_index="FEDFUND") for t, r in fedfund_swaps.items()]

fedfund_definition = CurveDefinition("USD.FEDFUND.1D")
assembly = MarketContext(AS_OF, market)                      # a mutable copy of the SOFR market
assembly["USD.FEDFUND.1D"] = assembly["USD.SOFR.1D"] + fedfund_definition.flat_on_pillars(assembly, fedfund_helpers)
fedfund_result = CurveCalibrator(fedfund_helpers, GlobalSolver(), fedfund_definition).calibrate(assembly)

market = fedfund_result.market
print(market)
print(f"converged: {fedfund_result.solver_result.converged}, max |residual| = {abs(fedfund_result.residuals).max():.1e}")
print("Fed Funds is built on the SOFR object:", market["USD.FEDFUND.1D"]["base"] is market["USD.SOFR.1D"])
pd.DataFrame({"pillar": fedfund_result.pillar_dates, "spread over SOFR (bp)": 1e4 * fedfund_result.zero_curve.params})

MarketContext(as_of=2026-10-16, curves=['USD.SOFR.1D', 'USD.STDCSA', 'USD.FEDFUND.1D'], frozen)
converged: True, max |residual| = 7.0e-15
Fed Funds is built on the SOFR object: True


,pillar,spread over SOFR (bp)
0,2027-10-20,-5.84
1,2028-10-20,-5.86
2,2029-10-22,-5.86
3,2031-10-20,-5.87
4,2036-10-20,-5.86


In [4]:
print(market.to_mermaid())

flowchart LR
    K0["USD.SOFR.1D"]
    K1["USD.STDCSA"]
    K1 -.->|same object| K0
    K2["USD.FEDFUND.1D"]
    K2 -->|base w=1| K0
    L0(["spread: ZeroCurve, 5 params"])
    K2 -->|spread w=1| L0


## 3. A book

Positive notional receives fixed. The compiled program prices against any market with the same
curve structure.

In [5]:
book = [
    Swap.fixed_float_swap(notional=100e6, rate_index="SOFR", fixed_rate=0.0370, tenor="5Y", as_of=AS_OF),
    Swap.fixed_float_swap(notional=-50e6, rate_index="SOFR", fixed_rate=0.0380, tenor="10Y", as_of=AS_OF),
    Swap.fixed_float_swap(notional=75e6, rate_index="FEDFUND", fixed_rate=0.0365, tenor="3Y", as_of=AS_OF),
]
positions = ("SOFR 5Y recv", "SOFR 10Y pay", "FF 3Y recv")
program = SwapPricer().compile(book)
pd.DataFrame({"position": positions, "PV": program.price(market).instrument_pv})

,position,PV
0,SOFR 5Y recv,-0.00
1,SOFR 10Y pay,"-83,058.67"
2,FF 3Y recv,"-84,755.13"


## 4. Risk against a template

The template is a strip of par swaps on both curves. It changes no PV; it re-expresses the book's
pillar risk as risk to those par rates, with the other template rates held. The report carries the
remainder the template cannot explain and both parallel DV01s (all pillars +1bp versus all
template quotes +1bp), which differ and are both shown.

In [6]:
sofr_tenors = ("1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y")
fedfund_tenors = ("2Y", "3Y", "5Y")
template = RiskTemplate(
    [swap_helper(rate=float("nan"), tenor=t, as_of=AS_OF) for t in sofr_tenors]
    + [swap_helper(rate=float("nan"), tenor=t, as_of=AS_OF, rate_index="FEDFUND") for t in fedfund_tenors],
    labels=tuple(f"SOFR {t}" for t in sofr_tenors) + tuple(f"FF {t}" for t in fedfund_tenors),
    name="SOFR par + FF basis",
)
engine = RiskEngine(program, market, template=template)
risk = engine.quote_risk(labels=positions)
risk.to_pandas().T

,SOFR 5Y recv,SOFR 10Y pay,FF 3Y recv
SOFR 1Y,0.00,0.77,2.72
SOFR 2Y,0.00,1.39,5.41
SOFR 3Y,0.00,3.22,8.32
SOFR 5Y,"-45,376.67",7.35,0.00
SOFR 7Y,0.00,13.47,0.00
SOFR 10Y,0.00,"41,545.38",0.00
SOFR 20Y,0.00,0.00,0.00
SOFR 30Y,0.00,0.00,0.00
FF 2Y,0.00,0.00,0.00
FF 3Y,0.00,0.00,"-21,188.78"


The Fed Funds swap shows SOFR risk on the SOFR template and its basis on the Fed Funds nodes; the
SOFR swaps carry nothing on the Fed Funds nodes. `pillar_parallel` moves every parameter of every
curve the template spans by 1bp, including the Fed Funds spread, so for the Fed Funds swap it is
SOFR risk plus basis risk; `template_parallel` moves every template quote by 1bp, which keeps the
spread where it is. Both are shown because neither is "the" DV01 without the qualifier.

In [7]:
bound = engine.bound()
print(f"condition of the template response : {bound.diagnostics.condition:,.1f}")
print(f"warnings                           : {bound.diagnostics.warnings or 'none'}")
pd.DataFrame({
    "node": bound.labels,
    "leverage (bp of base per bp of quote)": bound.diagnostics.leverage,
    "par rate %": 100 * bound.implied,
})

condition of the template response : 2.8
warnings                           : none


,node,leverage (bp of base per bp of quote),par rate %
0,SOFR 1Y,0.97,4.10
1,SOFR 2Y,0.99,3.85
2,SOFR 3Y,1.01,3.75
3,SOFR 5Y,1.03,3.70
4,SOFR 7Y,1.07,3.72
5,SOFR 10Y,1.12,3.78
6,SOFR 20Y,1.22,3.90
7,SOFR 30Y,1.51,3.88
8,FF 2Y,0.98,3.79
9,FF 3Y,1.01,3.69


### The hedge

Unit-instrument notionals (receive-fixed par swaps) that flatten each position on the template.
Negative means pay fixed.

In [8]:
pd.DataFrame(bound.hedge(risk) / 1e6, index=positions, columns=[f"{label} (mm)" for label in bound.labels]).T

,SOFR 5Y recv,SOFR 10Y pay,FF 3Y recv
SOFR 1Y (mm),0.00,0.01,0.03
SOFR 2Y (mm),0.00,0.01,0.03
SOFR 3Y (mm),0.00,0.01,0.03
SOFR 5Y (mm),-100.00,0.02,0.00
SOFR 7Y (mm),0.00,0.02,0.00
SOFR 10Y (mm),0.00,50.02,0.00
SOFR 20Y (mm),0.00,0.00,0.00
SOFR 30Y (mm),0.00,0.00,0.00
FF 2Y (mm),0.00,0.00,0.00
FF 3Y (mm),0.00,0.00,-75.00


### A scenario

`with_market` keeps the program and the template; the template rebinds because the curve moved.
A +1bp move of the SOFR curve carries the Fed Funds curve with it (it is built on the SOFR
object) and leaves the basis alone, so it matches the SOFR block of the pillar risk.

In [9]:
bumped = engine.with_market(market.bump("USD.SOFR.1D", 1e-4))       # +1bp on every SOFR pillar; Fed Funds follows
delta = bumped.program.price(bumped.market).instrument_pv - program.price(market).instrument_pv
sofr_block = engine.pillar_risk().block("USD.SOFR.1D").sum(axis=1) * 1e-4
pd.DataFrame({"position": positions, "PV change for +1bp SOFR": delta, "SOFR pillar DV01 from the report": sofr_block})

,position,PV change for +1bp SOFR,SOFR pillar DV01 from the report
0,SOFR 5Y recv,"-46,512.56","-46,523.91"
1,SOFR 10Y pay,"42,488.86","42,508.85"
2,FF 3Y recv,"-21,702.30","-21,705.53"
